# RupAI — M1 spike: Qwen-Image 2.1 on Colab T4

Goal: prove the model runs on a free Colab T4 and measure startup time, VRAM, RAM and speed.

1. **Runtime → Change runtime type → T4 GPU**
2. (Optional) add `HF_TOKEN` in **Colab Secrets** (key icon on the left). The repos are public, so it only helps download speed.
3. **Runtime → Run all**
4. When it finishes, copy the **M1 report** (last cells) and send it back.

In [ ]:
# @title 1. Config
MODEL_QUANT = "Q5_K_M"  # @param ["Q4_0", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]
TEXT_ENCODER = "int8_convrot"  # @param ["int8_convrot", "w4a8"]
RUN_BASELINE = True  # @param {type:"boolean"}
# M1b: compare Turbo LoRAs (cfg 1) against the 35-step baseline, same prompts and seed
PROMPTS = [
    "A blue ceramic mug on a wooden table by a window, soft morning light, product photography",
    "A smiling street food vendor at a night market in Phnom Penh, warm string lights, shallow depth of field, cinematic photo",
]
SEED = 42
# (label, LoRA file or None, steps, cfg)
VARIANTS = [
    ("turbo-8step", "p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors", 8, 1.0),
    ("turbo-6step", "p_q_image_2.1_6_steps_v0.2_comfyui_bf16.safetensors", 6, 1.0),
]
BASELINE = ("baseline-35step", None, 35, 5.0)  # runs last, first prompt only (~25 min on T4)

# Pinned versions (verified for Qwen-Image 2.1 nodes)
COMFY_TAG = "v0.46.1"
GGUF_NODES_COMMIT = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"
MODEL_REPO = "chatrlabs/Qwen-Image-2.1-Uncensored-Omni-Quantized"
MODEL_REV = "f5265c60e98818d5baa9e5329cea624daabcbece"
BASE_REPO = "Comfy-Org/Qwen-Image-2.1"
BASE_REV = "aacf31a6d48e707f352ecba714cee0680da3479b"

COMFY_DIR = "/content/ComfyUI"
OUTPUT_DIR = "/content/outputs"
LOG_DIR = "/content/logs"
COMFY_URL = "http://127.0.0.1:8188"

import os, time
T0 = time.time()
TIMINGS = {}
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"quant={MODEL_QUANT} text_encoder={TEXT_ENCODER}")

In [ ]:
# @title 2. Check GPU and RAM
import subprocess, psutil
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")
GPU_INFO = gpu.stdout.strip()
RAM_TOTAL_GB = psutil.virtual_memory().total / 1e9
DISK_FREE_GB = psutil.disk_usage("/content").free / 1e9
print("GPU :", GPU_INFO)
print(f"RAM : {RAM_TOTAL_GB:.1f} GB")
print(f"Disk: {DISK_FREE_GB:.0f} GB free")

In [ ]:
# @title 3. Install ComfyUI + ComfyUI-GGUF
import subprocess, sys, time
from pathlib import Path

t = time.time()

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return r.stdout

if not Path(COMFY_DIR).exists():
    sh(f"git clone -q --depth 1 --branch {COMFY_TAG} https://github.com/Comfy-Org/ComfyUI {COMFY_DIR}")
# Keep Colab's preinstalled torch/torchvision (requirements leave them unpinned)
sh(f"{sys.executable} -m pip install -q -r {COMFY_DIR}/requirements.txt websocket-client")

gguf_dir = Path(COMFY_DIR) / "custom_nodes" / "ComfyUI-GGUF"
if not gguf_dir.exists():
    sh(f"git clone -q https://github.com/city96/ComfyUI-GGUF {gguf_dir}")
    sh(f"git -C {gguf_dir} checkout -q {GGUF_NODES_COMMIT}")
sh(f"{sys.executable} -m pip install -q -r {gguf_dir}/requirements.txt")

# ComfyUI-GGUF does not list the 'qwen_image21' architecture yet.
# ComfyUI core already detects the model from its tensors, so we only add the name to the allow-list.
loader = gguf_dir / "loader.py"
src = loader.read_text()
if '"qwen_image21"' not in src:
    old = '"lumina2", "qwen_image"}'
    if old not in src:
        raise RuntimeError("ComfyUI-GGUF loader.py changed; patch needs updating")
    loader.write_text(src.replace(old, '"lumina2", "qwen_image", "qwen_image21"}'))
    print("Patched ComfyUI-GGUF: added qwen_image21")

TIMINGS["install_s"] = round(time.time() - t, 1)
print(f"Install done in {TIMINGS['install_s']} s")

In [ ]:
# @title 4. Download model files (parallel)
import os, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import hf_hub_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

UNET_FILE = f"qwen-image-2.1-step90000-{MODEL_QUANT}.gguf"
TE_FILE = f"qwen3vl_8b_{TEXT_ENCODER}.safetensors"
VAE_FILE = "qwen_image_2.1_vae_bf16.safetensors"

FILES = [
    (MODEL_REPO, MODEL_REV, UNET_FILE, "diffusion_models"),
    (BASE_REPO, BASE_REV, f"text_encoders/{TE_FILE}", "text_encoders"),
    (BASE_REPO, BASE_REV, f"vae/{VAE_FILE}", "vae"),
] + [(BASE_REPO, BASE_REV, f"loras/{lora}", "loras") for _, lora, _, _ in VARIANTS if lora]

def fetch(item):
    repo, rev, path, folder = item
    t = time.time()
    local = hf_hub_download(repo, path, revision=rev, token=HF_TOKEN, local_dir="/content/hf_models")
    dest = Path(COMFY_DIR) / "models" / folder / Path(path).name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        os.symlink(local, dest)
    size_gb = Path(local).stat().st_size / 1e9
    return Path(path).name, size_gb, time.time() - t

t = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    for name, size_gb, secs in pool.map(fetch, FILES):
        print(f"{name}: {size_gb:.2f} GB in {secs:.0f} s ({size_gb * 1000 / max(secs, 1):.0f} MB/s)")
TIMINGS["download_s"] = round(time.time() - t, 1)
print(f"Downloads done in {TIMINGS['download_s']} s")

In [ ]:
# @title 5. Start ComfyUI (background, 127.0.0.1 only)
import subprocess, sys, time, json, urllib.request

t = time.time()
comfy_log = open(f"{LOG_DIR}/comfyui.log", "a")
COMFY_PROC = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
     "--disable-auto-launch", "--output-directory", OUTPUT_DIR],
    cwd=COMFY_DIR, stdout=comfy_log, stderr=subprocess.STDOUT,
)

def get_json(path):
    with urllib.request.urlopen(COMFY_URL + path, timeout=5) as r:
        return json.loads(r.read())

for _ in range(300):
    if COMFY_PROC.poll() is not None:
        raise RuntimeError("ComfyUI exited. Run the 'Show logs' cell.")
    try:
        get_json("/system_stats")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ComfyUI did not start within 300 s. Run the 'Show logs' cell.")

info = get_json("/object_info")
for node in ["UnetLoaderGGUF", "TextEncodeQwenImage21", "QwenImage21Cache"]:
    if node not in info:
        raise RuntimeError(f"Node {node} missing. Run the 'Show logs' cell.")

TIMINGS["comfy_start_s"] = round(time.time() - t, 1)
TIMINGS["server_ready_s"] = round(time.time() - T0, 1)
print(f"ComfyUI ready in {TIMINGS['comfy_start_s']} s")
print(f"Time from Run all to server ready: {TIMINGS['server_ready_s']} s")

In [ ]:
# @title 6. Workflow + job runner
import copy, json, threading, time, uuid, subprocess, urllib.request
import psutil, websocket

# Same content as comfy/t2i_workflow.json in the repo
WORKFLOW = json.loads("{\n \"1\": {\n  \"class_type\": \"UnetLoaderGGUF\",\n  \"_meta\": {\n   \"title\": \"Transformer (GGUF)\"\n  },\n  \"inputs\": {\n   \"unet_name\": \"qwen-image-2.1-step90000-Q5_K_M.gguf\"\n  }\n },\n \"2\": {\n  \"class_type\": \"QwenImage21Cache\",\n  \"_meta\": {\n   \"title\": \"KV cache\"\n  },\n  \"inputs\": {\n   \"model\": [\n    \"1\",\n    0\n   ],\n   \"device\": \"auto\",\n   \"dtype\": \"default\"\n  }\n },\n \"3\": {\n  \"class_type\": \"CLIPLoader\",\n  \"_meta\": {\n   \"title\": \"Text encoder\"\n  },\n  \"inputs\": {\n   \"clip_name\": \"qwen3vl_8b_int8_convrot.safetensors\",\n   \"type\": \"qwen_image\",\n   \"device\": \"default\"\n  }\n },\n \"4\": {\n  \"class_type\": \"VAELoader\",\n  \"_meta\": {\n   \"title\": \"VAE\"\n  },\n  \"inputs\": {\n   \"vae_name\": \"qwen_image_2.1_vae_bf16.safetensors\"\n  }\n },\n \"5\": {\n  \"class_type\": \"TextEncodeQwenImage21\",\n  \"_meta\": {\n   \"title\": \"Prompt\"\n  },\n  \"inputs\": {\n   \"clip\": [\n    \"3\",\n    0\n   ],\n   \"prompt\": \"A blue ceramic mug on a wooden table by a window, soft morning light, product photography\",\n   \"negative_prompt\": \" \",\n   \"resolution\": 1024\n  }\n },\n \"6\": {\n  \"class_type\": \"EmptyLatentImage\",\n  \"_meta\": {\n   \"title\": \"Size\"\n  },\n  \"inputs\": {\n   \"width\": 1024,\n   \"height\": 1024,\n   \"batch_size\": 1\n  }\n },\n \"7\": {\n  \"class_type\": \"KSampler\",\n  \"_meta\": {\n   \"title\": \"Sampler\"\n  },\n  \"inputs\": {\n   \"model\": [\n    \"2\",\n    0\n   ],\n   \"positive\": [\n    \"5\",\n    0\n   ],\n   \"negative\": [\n    \"5\",\n    1\n   ],\n   \"latent_image\": [\n    \"6\",\n    0\n   ],\n   \"seed\": 42,\n   \"steps\": 35,\n   \"cfg\": 5.0,\n   \"sampler_name\": \"er_sde\",\n   \"scheduler\": \"simple\",\n   \"denoise\": 1.0\n  }\n },\n \"8\": {\n  \"class_type\": \"VAEDecode\",\n  \"_meta\": {\n   \"title\": \"Decode\"\n  },\n  \"inputs\": {\n   \"samples\": [\n    \"7\",\n    0\n   ],\n   \"vae\": [\n    \"4\",\n    0\n   ]\n  }\n },\n \"9\": {\n  \"class_type\": \"SaveImage\",\n  \"_meta\": {\n   \"title\": \"Save\"\n  },\n  \"inputs\": {\n   \"images\": [\n    \"8\",\n    0\n   ],\n   \"filename_prefix\": \"rupai\"\n  }\n }\n}")

class Monitor:
    """Polls GPU memory (nvidia-smi) and RAM every 0.5 s and keeps the peaks."""

    def __init__(self):
        self.peak_vram_mb = 0
        self.peak_ram_gb = 0.0
        self._stop = threading.Event()

    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                    capture_output=True, text=True,
                ).stdout
                self.peak_vram_mb = max(self.peak_vram_mb, int(out.strip().splitlines()[0]))
            except Exception:
                pass
            self.peak_ram_gb = max(self.peak_ram_gb, psutil.virtual_memory().used / 1e9)
            time.sleep(0.5)

    def __enter__(self):
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()


def build_workflow(prompt, width, height, seed, steps, cfg=5.0, negative=" ", lora=None):
    wf = copy.deepcopy(WORKFLOW)
    wf["1"]["inputs"]["unet_name"] = UNET_FILE
    if lora:
        # GGUF transformer -> LoRA -> KV cache
        wf["10"] = {
            "class_type": "LoraLoaderModelOnly",
            "_meta": {"title": "Turbo LoRA"},
            "inputs": {"model": ["1", 0], "lora_name": lora, "strength_model": 1.0},
        }
        wf["2"]["inputs"]["model"] = ["10", 0]
    wf["3"]["inputs"]["clip_name"] = TE_FILE
    wf["4"]["inputs"]["vae_name"] = VAE_FILE
    wf["5"]["inputs"]["prompt"] = prompt
    wf["5"]["inputs"]["negative_prompt"] = negative or " "
    wf["6"]["inputs"].update(width=width, height=height)
    wf["7"]["inputs"].update(seed=seed, steps=steps, cfg=cfg)
    return wf


def run_job(prompt, width, height, seed=42, steps=35, cfg=5.0, lora=None):
    client_id = uuid.uuid4().hex
    ws = websocket.create_connection(f"ws://127.0.0.1:8188/ws?clientId={client_id}", timeout=3600)
    wf = build_workflow(prompt, width, height, seed, steps, cfg, lora=lora)
    body = json.dumps({"prompt": wf, "client_id": client_id}).encode()
    req = urllib.request.Request(COMFY_URL + "/prompt", data=body, headers={"Content-Type": "application/json"})
    with Monitor() as mon:
        t_start = time.time()
        with urllib.request.urlopen(req) as r:
            prompt_id = json.loads(r.read())["prompt_id"]
        t_first_step = None
        while True:
            msg = ws.recv()
            if not isinstance(msg, str):
                continue  # binary preview frames
            m = json.loads(msg)
            data = m.get("data", {})
            if data.get("prompt_id") not in (None, prompt_id):
                continue
            if m["type"] == "progress":
                if t_first_step is None:
                    t_first_step = time.time()
                print(f"\r  step {data['value']}/{data['max']}  ({time.time() - t_start:.0f} s)", end="")
            elif m["type"] in ("execution_error", "execution_interrupted"):
                ws.close()
                raise RuntimeError(f"{m['type']}: {data.get('exception_type')}: {data.get('exception_message')}")
            elif m["type"] == "executing" and data.get("node") is None:
                break
        t_end = time.time()
    ws.close()
    print()
    with urllib.request.urlopen(f"{COMFY_URL}/history/{prompt_id}") as r:
        hist = json.loads(r.read())[prompt_id]
    img = hist["outputs"]["9"]["images"][0]
    path = f"{OUTPUT_DIR}/{img['subfolder']}/{img['filename']}".replace("//", "/")
    t_first_step = t_first_step or t_end
    sampling_s = t_end - t_first_step
    return {
        "size": f"{width}x{height}",
        "steps": steps,
        "total_s": round(t_end - t_start, 1),
        "before_sampling_s": round(t_first_step - t_start, 1),
        "s_per_step": round(sampling_s / max(steps - 1, 1), 2),
        "peak_vram_gb": round(mon.peak_vram_mb / 1024, 1),
        "peak_ram_gb": round(mon.peak_ram_gb, 1),
        "path": path,
    }

print("Runner ready")

In [ ]:
# @title 7. Compare Turbo LoRAs vs baseline (1024x1024, same seed)
from IPython.display import Image, display

RESULTS = []
runs = [(v, p) for v in VARIANTS for p in PROMPTS]
if RUN_BASELINE:
    runs.append((BASELINE, PROMPTS[0]))  # last: slowest
for (label, lora, steps, cfg), prompt in runs:
    pi = PROMPTS.index(prompt) + 1
    print(f"== {label} | prompt {pi}")
    try:
        r = run_job(prompt, 1024, 1024, seed=SEED, steps=steps, cfg=cfg, lora=lora)
        r.update(variant=label, prompt=pi, cfg=cfg)
        RESULTS.append(r)
        if "first_image_s" not in TIMINGS:
            TIMINGS["first_image_s"] = round(time.time() - T0, 1)
        print(f"   {r['total_s']} s total, {r['s_per_step']} s/step")
        display(Image(filename=r["path"], width=384))
    except Exception as e:
        print("   FAILED:", e)
        RESULTS.append({"variant": label, "prompt": pi, "error": str(e)})

In [ ]:
# @title 8. Side-by-side view (judge quality here)
import base64, html
from IPython.display import HTML, display

def tag(r):
    if "error" in r:
        return f"<div style='width:300px'>{html.escape(r['variant'])}<br>ERROR</div>"
    b64 = base64.b64encode(open(r["path"], "rb").read()).decode()
    cap = f"{r['variant']} - {r['steps']} steps, cfg {r['cfg']} - {r['total_s']} s"
    return f"<figure style='margin:4px'><img src='data:image/png;base64,{b64}' width='300'><figcaption>{html.escape(cap)}</figcaption></figure>"

for pi in range(1, len(PROMPTS) + 1):
    row = [tag(r) for r in RESULTS if r.get("prompt") == pi]
    display(HTML(f"<h4>Prompt {pi}</h4><div style='display:flex;flex-wrap:wrap'>" + "".join(row) + "</div>"))

In [ ]:
# @title 9. M1b report (copy everything below and send it back)
lines = ["## RupAI M1b report", ""]
lines.append(f"- GPU: {GPU_INFO}")
lines.append(f"- Config: {MODEL_QUANT} + text encoder {TEXT_ENCODER}, ComfyUI {COMFY_TAG}")
for k, v in TIMINGS.items():
    lines.append(f"- {k}: {v}")
lines.append("")
lines.append("| variant | prompt | steps | cfg | total s | before sampling s | s/step | peak VRAM GB | peak RAM GB |")
lines.append("|---|---|---|---|---|---|---|---|---|")
for r in RESULTS:
    if "error" in r:
        lines.append(f"| {r['variant']} | {r['prompt']} | | | ERROR: {r['error'][:80]} | | | | |")
        continue
    lines.append(
        f"| {r['variant']} | {r['prompt']} | {r['steps']} | {r['cfg']} | {r['total_s']} | "
        f"{r['before_sampling_s']} | {r['s_per_step']} | {r['peak_vram_gb']} | {r['peak_ram_gb']} |"
    )
lines.append("")
lines.append("Quality (fill in): turbo-8step = ? / turbo-6step = ? vs baseline (good / ok / bad)")
report = "\n".join(lines)
print(report)
with open(f"{LOG_DIR}/m1b_report.md", "w") as f:
    f.write(report)

In [ ]:
# @title 10. Show logs (last 50 lines)
!tail -n 50 /content/logs/comfyui.log